# SP-005 — M5 Best Residual Fit

**Autor:** Luis Anthony Ibáñez Herrera  
**Estado:** evidencia reproducible; revisión formal de Andro Joseph Quispe Cesias pendiente.

Este cuaderno presenta los resultados producidos por `experimento_m5.py`. FF, BF, WF y M5 cargan los fixtures oficiales TA-027 y comparten el motor, el rasterizador, la geometría y las métricas versionadas del backend. El cuaderno ya no incorpora copias del algoritmo.

La ejecución que generó el artefacto guardado se hizo localmente en Windows con Python 3.11.15. No se afirma que esta revisión haya sido ejecutada en Google Colab. El formato se puede abrir allí para consultar estos resultados; para reproducirlos en Colab, el código de abajo clona la rama del spike.

## 1. Entorno y procedencia

La celda siguiente informa el runtime que ejecuta el cuaderno. La procedencia de los datos está registrada en `resultados/resumen_comparativo.json`, incluido el commit del runner, la versión de Python y el número de repeticiones.

In [1]:
# Registrar el runtime real que ejecuta esta celda.
import platform
import sys

print('Python:', sys.version.splitlines()[0])
print('Plataforma:', platform.platform())
print('Runtime Google Colab:', 'google.colab' in sys.modules)

Python: 3.11.15 (main, Jun 23 2026, 15:20:37) [MSC v.1944 64 bit (AMD64)]
Plataforma: Windows-10-10.0.26200-SP0
Runtime Google Colab: False


## 2. Reproducción de las suites

Desde la raíz del repositorio, ejecutar el runner genera tres repeticiones para cada combinación de método y caso (36 ejecuciones), guarda cada muestra y calcula la mediana de tiempo:

```bash
python docs/spikes/sp-005-heuristica-m5/experimento_m5.py --repetitions 3
```

En Colab, la siguiente celda prepara una copia de la rama pública del spike si el repositorio no está disponible en el directorio actual. Después carga el resumen generado y comprueba determinismo. Para medir en Colab, ejecutar primero el comando del runner en una celda de shell; los tiempos guardados en este cuaderno corresponden a la ejecución local identificada en la evidencia.

In [2]:
import json
import subprocess
from pathlib import Path

# Usar el checkout actual o clonar la rama del spike para Colab.
def find_repository(start):
    for candidate in (start, *start.parents):
        runner = candidate / 'docs/spikes/sp-005-heuristica-m5/experimento_m5.py'
        if runner.is_file():
            return candidate
    return None

repo = find_repository(Path.cwd())
if repo is None:
    repo = Path('/content/optimizador-vidrio')
    if not (repo / 'docs/spikes/sp-005-heuristica-m5/experimento_m5.py').is_file():
        subprocess.run([
            'git', 'clone', '--depth', '1', '--branch', 'spike/SP-005-heuristica-m5',
            'https://github.com/melendezzz26/optimizador-vidrio.git', str(repo)
        ], check=True)

# Leer artefactos generados por el runner; no recalcular métricas en el notebook.
result_dir = repo / 'docs/spikes/sp-005-heuristica-m5/resultados'
summary = json.loads((result_dir / 'resumen_comparativo.json').read_text(encoding='utf-8'))
assert summary['repeticiones_por_metodo_y_caso'] >= 3
assert summary['determinismo_funcional_completo']
print('Fuente: backend versionado + loader y fixtures oficiales TA-027')
print('Commit del runner:', summary['commit_base_runner'])
print('Python de la ejecución registrada:', summary['python'])
print('Repeticiones por método/caso:', summary['repeticiones_por_metodo_y_caso'])
print('Determinismo funcional en las 12 combinaciones: PASS')

print('\nCASO             MÉTODO  PZ   RET PLAN USO %   MERMA     RECUP.   MEDIANA ms  MUESTRAS ms')
for case_id, methods in summary['casos'].items():
    for code, entry in methods.items():
        result = entry['resultados_funcionales']
        metrics = result['metrics']
        remnant_ids = {(p['material_id'], p['stock_unit_index']) for p in result['placements'] if p['stock_source'] == 'RETAZO'}
        retazos = len(remnant_ids)
        print(f"{case_id:16} {code:5} {result['placed_units']:>2}/{result['total_units']:<2} {retazos:>3} "
              f"{metrics['new_sheets_used']:>4} {float(metrics['utilization_pct']):>6.3f} "
              f"{float(metrics['waste_mm2']):>8.2f} {float(metrics['recoverable_remnant_mm2']):>8.2f} "
              f"{entry['tiempo_ms_mediana']:>10.2f}  {entry['tiempo_ms_muestras']}")

Fuente: backend versionado + loader y fixtures oficiales TA-027
Commit del runner: 42ac079e705c079a61bbca37d70b0f593b52dd77
Python de la ejecución registrada: 3.11.15 (main, Jun 23 2026, 15:20:37) [MSC v.1944 64 bit (AMD64)]
Repeticiones por método/caso: 3
Determinismo funcional en las 12 combinaciones: PASS

CASO             MÉTODO  PZ   RET PLAN USO %   MERMA     RECUP.   MEDIANA ms  MUESTRAS ms
EXP-SMALL-001    FF     3/3    1    1 60.000     0.00     8.00      95.03  [95.03, 104.5, 93.55]
EXP-SMALL-001    BF     3/3    1    1 60.000     0.00     8.00      96.76  [98.3, 96.76, 96.42]
EXP-SMALL-001    WF     3/3    1    1 60.000     0.00     8.00      97.37  [96.18, 97.66, 97.37]
EXP-SMALL-001    M5     3/3    1    1 60.000     0.00     8.00      96.94  [96.94, 95.65, 97.4]
EXP-MEDIUM-001   FF     9/9    2    1 28.413   201.86    58.00     181.06  [181.06, 184.41, 176.41]
EXP-MEDIUM-001   BF     9/9    2    1 28.413   193.86    66.00     243.34  [246.9, 238.82, 243.34]
EXP-MEDIUM-001

## 3. Interpretación

- `EXP-SMALL-001`: paridad funcional entre los cuatro métodos.
- `EXP-MEDIUM-001`: M5 conserva 170 mm² recuperables (20 mm² más que WF) y reduce su merma en 20 mm²; usa dos planchas nuevas, igual que WF y una más que FF/BF.
- `EXP-COMPLEX-001`: M5 conserva 318 mm² recuperables (50 mm² más que WF) y reduce su merma en 50 mm²; usa cuatro planchas, igual que WF y una más que FF/BF.

La evidencia respalda un **GO condicionado** para continuar evaluando M5 cuando priorizar retazos recuperables compense abrir más planchas. No demuestra superioridad general frente a FF/BF: en Medium y Complex, FF/BF usan menos planchas y generan menos merma. La revisión de Andro continúa pendiente.